# Laboratorio 01. Tensores y autograd

**Pregunta central:** ¿Cómo representa PyTorch los datos y calcula derivadas de una expresión?

## Objetivos

- Inspeccionar forma, tipo y dispositivo de tensores.
- Convertir entre NumPy y PyTorch sin confundir memoria compartida.
- Contrastar autograd con una derivada analítica y diferencias finitas.

## Prerrequisitos y conexión

Usa la relación entre datos y parámetros del laboratorio 00. Se requieren operaciones elementales con vectores y derivadas de una variable.

## Teoría breve

Autograd construye un grafo de operaciones y aplica la regla de la cadena. En una función escalar $L(w)$, `backward()` deja $\partial L/\partial w$ en `w.grad`.

## Problema y datos

Inspeccionamos un arreglo local pequeño y una función escalar con parámetros definidos en el notebook; no se requieren descargas.

## Evaluación

Compara el gradiente calculado por `backward()` con el gradiente analítico y una diferencia central. La discrepancia numérica, la forma y el dispositivo son las comprobaciones de este laboratorio.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Inspeccionar tensores
Observamos forma, tipo, dispositivo y el paso NumPy → PyTorch.


In [ ]:
array = np.arange(12, dtype=np.float32).reshape(4, 3)
tensor = torch.from_numpy(array).to(device)
print('forma:', tensor.shape, '| dtype:', tensor.dtype, '| device:', tensor.device)
print(tensor)
print('¿Comparten memoria antes de mover a GPU?:', torch.from_numpy(array).data_ptr() == torch.from_numpy(array).data_ptr())


**Resultado e interpretación.** La forma describe dimensiones, dtype controla precisión y device indica dónde se ejecuta la operación. La conversión CPU puede compartir memoria; el movimiento a GPU crea almacenamiento en otro dispositivo.


### Verificar una derivada con autograd y diferencias finitas

Calculamos una derivada de una función escalar de pesos por dos vías. La diferencia finita es una comprobación numérica, no el método que usa PyTorch durante el entrenamiento.

In [ ]:
weights = torch.tensor([0.2, -0.4], dtype=torch.float64, requires_grad=True)
inputs = torch.tensor([1.5, -2.0], dtype=torch.float64)
target = torch.tensor(0.7, dtype=torch.float64)
loss = 0.5 * (torch.dot(weights, inputs) - target) ** 2
loss.backward()
autograd_gradient = weights.grad.detach().numpy()

def scalar_loss(values):
    return 0.5 * (np.dot(values, inputs.numpy()) - target.item()) ** 2

values = weights.detach().numpy().copy()
epsilon = 1e-6
finite_difference = np.array([
    (scalar_loss(values + np.eye(2)[index] * epsilon) - scalar_loss(values - np.eye(2)[index] * epsilon)) / (2 * epsilon)
    for index in range(2)
])
print('gradiente autograd:', autograd_gradient.round(6))
print('diferencia finita:', finite_difference.round(6))
assert np.allclose(autograd_gradient, finite_difference, atol=1e-6)

**Interpretación.** Las dos aproximaciones deben coincidir dentro de la tolerancia numérica. Al cambiar `epsilon`, el error puede crecer por redondeo o por aproximación demasiado gruesa.

## Limitaciones y conclusiones

La diferencia finita aproxima una derivada y depende del tamaño del paso; `autograd` aplica la regla de la cadena al grafo de operaciones registrado. Un gradiente correcto no implica que el modelo, la función de pérdida o los datos sean adecuados.

## Ejercicios

1. **Guiado:** cambia un componente de `weights` y compara el gradiente automático con una diferencia central para ese componente.
2. **Independiente:** crea un tensor de forma `(2, 3, 4)` y explica qué representan sus tres dimensiones en un ejemplo propio.
3. **Desafío:** construye una expresión con dos tensores hoja y verifica qué gradientes quedan acumulados tras llamar `backward()` dos veces.

## Conexión con el siguiente laboratorio

Con los productos y derivadas definidos, el siguiente laboratorio calcula una neurona y su frontera de decisión.